# CassavaGuard — EfficientNet-B2 + saliency_crop pipeline on Google Colab

Notebook นี้เรียก `backend/training/train_cnn_torch.py` (PyTorch) จากโปรเจกต์โดยตรงด้วย `--pipeline saliency_crop` ซึ่งเป็น pipeline ที่ดีที่สุดจากการทดลอง 8 แบบที่บันทึกไว้ใน `docs/superpowers/specs/2026-08-24-cbb-recall-experiments-report.md`:

- TFDS `cassava:0.1.0` official train/validation/test splits, real data only
- exact-pixel + perceptual (dHash/pHash) duplicate quarantine
- `SaliencyGuidedCrop`: deterministic classical-CV crop toward the sub-window with the most local color anomaly (real pixels only, no trained segmentation/attribution model)
- EfficientNet-B2 ImageNet transfer learning + inverse-frequency class weights
- checkpoint selection from validation macro-F1 only, test opened after selection
- ONNX export + PyTorch/ONNX parity check
- optional 4-view flip TTA evaluation via `evaluate_cnn_tta_pipeline.py`

> **ผลอ้างอิงจากเครื่อง local (Apple Silicon MPS):** test accuracy 85.53% / macro-F1 81.24% / CBB recall 68.18% (no TTA); กับ TTA: accuracy 86.28% / macro-F1 82.37% / **CBB recall 69.48%** (ดีกว่า production 67.53%) แต่ยังต่ำกว่า production overall (88.20%/83.63%) — ยังไม่ผ่านเกณฑ์โปรโมต ใช้ notebook นี้เพื่อรีรันยืนยันผลบน GPU อื่น หรือทดลองต่อ ไม่ใช่เพื่อ deploy ตรง ๆ

## 1. เตรียมโปรเจกต์ก่อนเปิด Notebook

เลือกวิธีใดวิธีหนึ่งใน cell ตั้งค่า:

1. `upload_zip` — ZIP โปรเจกต์แล้วอัปโหลดเข้า Colab
2. `drive` — เก็บโฟลเดอร์โปรเจกต์ไว้ใน Google Drive
3. `git` — clone จาก Git repository

คำสั่งสร้าง ZIP แบบไม่รวมไฟล์ใหญ่ในเครื่อง local:

```bash
cd /Users/norapol/Documents
zip -r cassavaguard-colab.zip cassavaguard \
  -x 'cassavaguard/backend/training/.venv*/*' \
     'cassavaguard/.venv/*' \
     'cassavaguard/.venv-audit/*' \
     'cassavaguard/node_modules/*' \
     'cassavaguard/.git/*' \
     'cassavaguard/database/*' \
     'cassavaguard/uploads/*' \
     'cassavaguard/tmp/*'
```

In [ ]:
#@title 2. ตั้งค่า Training
SOURCE_MODE = "upload_zip"  #@param ["upload_zip", "drive", "git"]
GIT_REPO_URL = "https://github.com/norapolamarit-commits/cassavaguard-render.git"  #@param {type:"string"}
GIT_BRANCH = "main"  #@param {type:"string"}
DRIVE_PROJECT_PATH = "/content/drive/MyDrive/cassavaguard"  #@param {type:"string"}
TFDS_DATA_DIR = "/content/tensorflow_datasets"  #@param {type:"string"}
PIPELINE = "saliency_crop"  #@param ["none", "color_constancy", "leaf_crop", "tiled_crop", "saliency_crop"]
ARCHITECTURE = "efficientnet_b2"  #@param ["efficientnet_b0", "efficientnet_b2", "efficientnet_b3", "mobilenet_v3_large"]
IMAGE_SIZE = 260  #@param {type:"integer"}
EPOCHS_HEAD = 5  #@param {type:"integer"}
EPOCHS_FINE = 14  #@param {type:"integer"}
FINE_TUNE_BLOCKS = 4  #@param {type:"integer"}
BATCH_SIZE = 32  #@param {type:"integer"}
PATIENCE = 4  #@param {type:"integer"}
SEED = 42  #@param {type:"integer"}
REQUIRE_GPU = True  #@param {type:"boolean"}
RUN_TTA_EVAL = True  #@param {type:"boolean"}
SAVE_ARTIFACTS_TO_DRIVE = True  #@param {type:"boolean"}
DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/CassavaGuard/models"  #@param {type:"string"}

assert SOURCE_MODE in {"upload_zip", "drive", "git"}
assert EPOCHS_HEAD >= 0 and EPOCHS_FINE >= 0 and EPOCHS_HEAD + EPOCHS_FINE > 0
assert min(FINE_TUNE_BLOCKS, BATCH_SIZE, PATIENCE, IMAGE_SIZE) > 0

In [ ]:
#@title 3. นำโปรเจกต์เข้า Colab
from pathlib import Path
import shutil
import subprocess
import sys

if SOURCE_MODE == "drive":
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DIR = Path(DRIVE_PROJECT_PATH).resolve()
elif SOURCE_MODE == "git":
    if not GIT_REPO_URL:
        raise ValueError("กรุณากำหนด GIT_REPO_URL")
    PROJECT_DIR = Path("/content/cassavaguard")
    if PROJECT_DIR.exists():
        shutil.rmtree(PROJECT_DIR)
    subprocess.run(["git", "clone", "--depth", "1", "--branch", GIT_BRANCH,
                    GIT_REPO_URL, str(PROJECT_DIR)], check=True)
else:
    from google.colab import files
    uploaded = files.upload()
    zip_names = [name for name in uploaded if name.lower().endswith(".zip")]
    if len(zip_names) != 1:
        raise ValueError("กรุณาอัปโหลด ZIP ของโปรเจกต์จำนวน 1 ไฟล์")
    extract_dir = Path("/content/cassavaguard-upload")
    if extract_dir.exists():
        shutil.rmtree(extract_dir)
    extract_dir.mkdir(parents=True)
    shutil.unpack_archive(f"/content/{zip_names[0]}", extract_dir)
    matches = list(extract_dir.rglob("backend/training/train_cnn_torch.py"))
    if len(matches) != 1:
        raise RuntimeError(f"หา backend/training/train_cnn_torch.py ไม่พบหรือพบซ้ำ: {matches}")
    PROJECT_DIR = matches[0].parents[2]

required = [
    PROJECT_DIR / "requirements-training-torch.txt",
    PROJECT_DIR / "backend/training/train_cnn_torch.py",
    PROJECT_DIR / "backend/training/evaluate_cnn_tta_pipeline.py",
    PROJECT_DIR / "backend/training/verify_artifacts.py",
]
missing = [str(path) for path in required if not path.is_file()]
if missing:
    raise FileNotFoundError(f"โปรเจกต์ไม่ครบ: {missing}")
print("PROJECT_DIR =", PROJECT_DIR)

In [ ]:
#@title 4. ติดตั้ง Training Dependencies (PyTorch)
import subprocess
import sys

subprocess.run([sys.executable, "-m", "pip", "install", "--quiet",
                "-r", str(PROJECT_DIR / "requirements-training-torch.txt")], check=True)
print("ติดตั้ง dependencies สำเร็จ")
print("หาก Colab แจ้งให้ Restart session ให้กด Restart แล้วรันใหม่ตั้งแต่ cell 2")

In [ ]:
#@title 5. ตรวจ GPU และ Environment
import json
import platform
import torch
import torchvision

environment = {
    "python": platform.python_version(),
    "torch": torch.__version__,
    "torchvision": torchvision.__version__,
    "cuda_available": torch.cuda.is_available(),
    "cuda_device": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
}
print(json.dumps(environment, indent=2))
if REQUIRE_GPU and not torch.cuda.is_available():
    raise RuntimeError("ไม่พบ GPU: ไปที่ Runtime > Change runtime type > T4 GPU")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
#@title 6. เตรียม TFDS Cassava (รูปแบบ cassavaleafdata จาก tensorflow_datasets)
from pathlib import Path
import subprocess, sys

Path(TFDS_DATA_DIR).mkdir(parents=True, exist_ok=True)
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "tensorflow-datasets"], check=True)
import tensorflow_datasets as tfds
builder = tfds.builder("cassava", data_dir=TFDS_DATA_DIR)
builder.download_and_prepare()
extracted = list(Path(TFDS_DATA_DIR, "downloads", "extracted").glob("*/cassavaleafdata"))
if len(extracted) != 1:
    raise RuntimeError(f"หา cassavaleafdata ไม่พบหรือพบซ้ำ: {extracted}")
DATA_DIR = extracted[0]
print("DATA_DIR =", DATA_DIR)

In [ ]:
#@title 7. Train EfficientNet-B2 + saliency_crop pipeline
import subprocess
import sys
import time

OUTPUT_DIR = PROJECT_DIR / f"tmp/candidates/{ARCHITECTURE}_pipeline_{PIPELINE}_colab"
command = [
    sys.executable, str(PROJECT_DIR / "backend/training/train_cnn_torch.py"),
    "--architecture", ARCHITECTURE,
    "--image-size", str(IMAGE_SIZE),
    "--device", DEVICE,
    "--pipeline", PIPELINE,
    "--epochs-head", str(EPOCHS_HEAD),
    "--epochs-fine", str(EPOCHS_FINE),
    "--fine-tune-blocks", str(FINE_TUNE_BLOCKS),
    "--batch-size", str(BATCH_SIZE),
    "--patience", str(PATIENCE),
    "--seed", str(SEED),
    "--data-dir", str(DATA_DIR),
    "--output-dir", str(OUTPUT_DIR),
]
print("Running:", " ".join(command))
started = time.time()
subprocess.run(command, cwd=PROJECT_DIR, check=True)
print(f"Training completed in {(time.time() - started) / 60:.1f} minutes")
print("OUTPUT_DIR =", OUTPUT_DIR)

In [ ]:
#@title 8. (ถ้าเปิด RUN_TTA_EVAL) ประเมินด้วย 4-view flip TTA
import subprocess
import sys

ONNX_PATH = OUTPUT_DIR / f"cnn_{ARCHITECTURE}.onnx"
METRICS_PATH = OUTPUT_DIR / f"cnn_{ARCHITECTURE}_metrics.json"

if RUN_TTA_EVAL:
    if PIPELINE == "tiled_crop":
        print("tiled_crop เป็น train-only (stochastic) ไม่มีรูปแบป deterministic สำหรับ TTA -- ข้ามขั้นนี้")
    else:
        tta_command = [
            sys.executable, "-m", "backend.training.evaluate_cnn_tta_pipeline",
            "--data-dir", str(DATA_DIR),
            "--model", str(ONNX_PATH),
            "--metrics", str(METRICS_PATH),
            "--pipeline", PIPELINE,
        ]
        subprocess.run(tta_command, cwd=PROJECT_DIR, check=True)
else:
    print("RUN_TTA_EVAL=False, ข้ามขั้นนี้")

In [ ]:
#@title 9. แสดงผล Validation/Test (เทียบกับ production baseline)
import json

metrics = json.loads(METRICS_PATH.read_text())
summary = {
    "model_id": metrics["model_id"],
    "pipeline": metrics["training"].get("pipeline"),
    "production_eligible": metrics.get("production_eligible"),
    "effective_split_counts": metrics["dataset"]["effective_split_counts"],
    "validation_macro_f1": metrics["validation"]["macro_f1"],
    "test_accuracy": metrics["test"]["accuracy"],
    "test_macro_f1": metrics["test"]["macro_f1"],
    "test_cbb": metrics["test"]["per_class"]["cbb"],
    "onnx_parity": metrics["onnx_parity"],
}
if "tta_test" in metrics:
    summary["tta_test_accuracy"] = metrics["tta_test"]["accuracy"]
    summary["tta_test_macro_f1"] = metrics["tta_test"]["macro_f1"]
    summary["tta_test_cbb"] = metrics["tta_test"]["per_class"]["cbb"]
print(json.dumps(summary, indent=2, ensure_ascii=False))
print("\nProduction baseline reference: test accuracy 88.20%, macro-F1 83.63%, CBB recall 67.53% (with TTA)")

In [ ]:
#@title 10. บันทึกและดาวน์โหลด Artifacts
from pathlib import Path
import shutil
import zipfile

artifact_names = [ONNX_PATH.name, METRICS_PATH.name]
for path in (ONNX_PATH, METRICS_PATH):
    if not path.is_file():
        raise FileNotFoundError(path)

archive = Path(f"/content/cassavaguard_{ARCHITECTURE}_{PIPELINE}_artifacts.zip")
with zipfile.ZipFile(archive, "w", compression=zipfile.ZIP_DEFLATED) as bundle:
    for path in (ONNX_PATH, METRICS_PATH):
        bundle.write(path, arcname=path.name)

if SAVE_ARTIFACTS_TO_DRIVE:
    from google.colab import drive
    if not Path("/content/drive/MyDrive").exists():
        drive.mount("/content/drive")
    drive_output = Path(DRIVE_OUTPUT_DIR)
    drive_output.mkdir(parents=True, exist_ok=True)
    shutil.copy2(archive, drive_output / archive.name)
    for path in (ONNX_PATH, METRICS_PATH):
        shutil.copy2(path, drive_output / path.name)
    print("Saved to", drive_output)

from google.colab import files
files.download(str(archive))

## 11. สิ่งสำคัญก่อนนำโมเดลกลับเข้า production

นี่คือ **candidate**, ไม่ใช่ production model ตามกฎของโครงการ:

- ต้องดีกว่า production ทั้ง **overall metric** (accuracy/macro-F1) และ **CBB recall** พร้อมกัน จึงจะโปรโมตได้
- จากผล local (Apple MPS) ที่บันทึกไว้: `saliency_crop` + TTA ได้ CBB recall ดีขึ้นแต่ overall accuracy/macro-F1 ยังต่ำกว่า production — ยังไม่ผ่านเกณฑ์โปรโมต
- ดูรายละเอียดการทดลองทั้ง 8 ตัวที่ `docs/superpowers/specs/2026-08-24-cbb-recall-experiments-report.md`

ห้าม เปิด `USE_CNN=true` ใน production จากคะแนน candidate นี้โดยพลการ ต้องผ่าน `verify_artifacts.py`, `quality_gate.py` และ independent Thai-field validation ก่อนเสมอ ดู `docs/TRAINING.md`